In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
cdf = spark.read.csv("cust3.csv", header=True, inferSchema=True)
pdf = spark.read.csv("prod3.csv", header=True, inferSchema=True)
odf = spark.read.csv("ord3.csv", header=True, inferSchema=True)

In [ ]:
cdf.createOrReplaceTempView("cust")
pdf.createOrReplaceTempView("prod")
odf.createOrReplaceTempView("ord")


In [ ]:
spark.sql("""select * from cust""").show()

+-----------+------+---------+---+-----------------+
|customer_id|  name|     city|age|registration_date|
+-----------+------+---------+---+-----------------+
|          1|  Amit|    Delhi| 25|       2023-01-10|
|          2|  Sara|   Mumbai| 30|       2022-12-05|
|          3|  John|Bangalore| 28|       2023-02-15|
|          4|   Ali|Hyderabad| 35|       2021-11-20|
|          5|  Neha|  Chennai| 27|       2023-03-01|
|          6|  Ravi|    Delhi| 40|       2020-07-19|
|          7| Priya|   Mumbai| 22|       2023-04-12|
|          8| Kiran|Hyderabad| 31|       2022-05-22|
|          9| David|Bangalore| 29|       2023-01-25|
|         10| Meena|  Chennai| 33|       2021-09-10|
|         11| Arjun|    Delhi| 26|       2022-08-18|
|         12|Fatima|Hyderabad| 24|       2023-02-28|
|         13| Rahul|   Mumbai| 37|       2020-06-30|
|         14| Sneha|  Chennai| 21|       2023-03-14|
|         15|Vikram|Bangalore| 45|       2019-12-11|
|         16|  Zoya|    Delhi| 23|       2023-

top 3 custs based on their spending

In [ ]:
spark.sql("""

with spending_cte as (
      select c.customer_id,c.name,sum(o.quantity*p.price) as spending
      from
      cust c join ord o
        on c.customer_id = o.customer_id
      join prod p
        on p.product_id = o.product_id
      group by c.customer_id,c.name
)
select customer_id,name,spending
from
(select *,dense_rank() over(order by spending desc) as rank from spending_cte)x
where rank <=3

""").show()

+-----------+----+--------+
|customer_id|name|spending|
+-----------+----+--------+
|          4| Ali|  140000|
|          1|Amit|   74000|
|          2|Sara|   63000|
+-----------+----+--------+



In [ ]:
spark.sql("""
WITH spending_cte AS (
    SELECT
        c.customer_id,
        c.name,
        SUM(o.quantity * p.price) AS spending
    FROM cust c
    JOIN ord o
        ON c.customer_id = o.customer_id
    JOIN prod p
        ON p.product_id = o.product_id
    GROUP BY c.customer_id, c.name
),
ranked AS (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY spending DESC) AS rank
    FROM spending_cte
)

SELECT
    customer_id,
    name,
    spending
FROM ranked
WHERE rank <= 3
""").show()

+-----------+----+--------+
|customer_id|name|spending|
+-----------+----+--------+
|          4| Ali|  140000|
|          1|Amit|   74000|
|          2|Sara|   63000|
+-----------+----+--------+

